# Evaluating the tutor: a data scientist's walkthrough

*For learning. Nothing here runs in CI, and nothing costs money: it replays recordings of real runs.*

## What's going on, in one paragraph

The tutor in "Ask about this lesson" now has **prepared answers**: 1,728 answers to its standard
questions, written ahead of time by a strong AI model, so Ask works instantly even when no AI is running.
A second AI model (the **judge**) checked every answer before it went live. Now we'd like a **cheaper**
judge to keep checking answers every day. Before trusting it, we test it, the way a data scientist would
test any measuring tool: is it accurate, is it consistent, how sure are we, and is it worth the money?

## Words you'll meet

| Word | Plain meaning |
|---|---|
| **Judge** | An AI model asked to grade another AI's answer against a checklist |
| **Reference** | The judge we compare against (here Sonnet). Not the truth: just another opinion |
| **Ground truth / golden set** | Answers a **person** has graded. The closest thing to the truth we have |
| **Dev / test / holdout** | Practice set (try things), exam set (check once before a release), locked set (big decisions only) |
| **Stratified sample** | A sample that deliberately includes every group in fair amounts, instead of pure luck |
| **Confidence interval** | The range the true value probably sits in. Narrow means sure; wide means unsure |
| **Kappa / AC1** | How much two judges agree, beyond what luck alone would give |
| **Bootstrap** | Re-running the analysis on many shuffled copies of the data to see how much the result wobbles |
| **p-value** | How surprising a result would be if nothing were really going on. Small means "probably real" |

## How to use this notebook

Run the cells from top to bottom (Shift + Enter). Cells that start with `# TODO` are **exercises**: write
the function, then run the test cell below it. Green means right. The answers are in
`evaluating_the_tutor_answers.py` beside this notebook, if you get stuck.

In [ ]:
import json
import math
import random
from collections import Counter
from pathlib import Path

import ipytest
import pandas as pd
import pytest

from drill import tutor_prepared as tp
from drill import tutor_prepared_eval as tpe
from drill.content.build import build_data

ipytest.autoconfig(clean="test_*")

DATA = build_data()                      # every lesson, as the page sees it
ANSWERS = tp.load()                      # the prepared answers: {lesson_id: {question key: record}}
CHIPS = tp.chips(DATA)                   # the standard questions, as the page asks them
RECORDINGS = Path(tp.__file__).parent / "content" / "prepared_eval"
REC = json.loads((RECORDINGS / "dev_sample_haiku.json").read_text())
PHASE = tpe.phases(DATA)                 # each lesson's phase (or the platform track)

print(f"{sum(len(v) for v in CHIPS.values())} questions across {len(CHIPS)} lessons")
print(f"{sum(len(v) for v in ANSWERS.values())} prepared answers; status: {Counter(r['status'] for v in ANSWERS.values() for r in v.values())}")
print(f"recorded judge run: {REC['params']['judge_model']} on {len(REC['answers'])} answers")

**What that cell did.** It loaded four things: every lesson (`DATA`), every prepared answer (`ANSWERS`),
the list of standard questions (`CHIPS`), and a **recording** (`REC`) of the real run where Haiku judged a
sample of answers. Everything below works on these.

## 0. Write the plan before looking at results

> **In plain words.** Decide what "good enough" means *before* you see the numbers. Otherwise it's too
> easy to move the goalposts after a disappointing result. The course calls this pre-registration
> (lesson "Write the plan before the data").

Honest note: the first Haiku run happened before this plan was written. So treat that run as
**exploration**, and this plan as the rule for the **next** runs (a new judge prompt, or a new judge).

In [ ]:
PLAN = {
    "question": "Can a cheaper judge stand in for careful checking of the tutor's prepared answers, every day?",
    "ground_truth": "a person's verdict on a golden set (section 11); until then, Sonnet as a stand-in, said out loud",
    "primary_metric": "agreement with the ground truth on answers that should FAIL (recall) and on answers that should PASS (specificity)",
    "decision_rule": "trust the judge for daily monitoring only if recall >= 0.8 AND specificity >= 0.8, "
                     "each with the lower end of its lesson-level bootstrap interval >= 0.7",
    "secondary": ["Gwet's AC1 and Cohen's kappa", "cost per answer", "consistency when run twice"],
    "comparisons": "a new judge prompt is compared with McNemar on the same answers; many group-level tests are Holm-adjusted",
    "splits": "tune on dev (or its sample); report on test once; never look at holdout",
}
for k, v in PLAN.items():
    print(f"{k:15} {v}")

**What the rule means, and what it doesn't.** The two numbers are not "the judge is right 80% of the
time". *Recall* asks: of the answers the reference failed, did the judge fail at least 4 in 5? *Specificity*
asks: of the answers the reference passed, did it pass at least 4 in 5? A lucky 80% doesn't count: the low
end of the interval must stay above 70%, which bites when a sample holds only a few dozen failures.

**Why 80 and not 100.** No judge agrees with anything all the time, not even with itself (section 11 runs
one twice). Two careful people disagree too, and how often they agree is the ceiling no judge can beat
(section 14). So 80 is a floor for one job: **daily monitoring**, where the judge flags answers for a
person to look at and never has the final say. For deciding on its own what gets published, 80 would be
far too low; that job needs a person, or a much tighter bar.

**The number only buys trust if the reference deserves it.** Against Sonnet, 80% means "agrees with
another unchecked judge 4 times in 5", which says nothing about whether either is right. Against a
person's grades it is a claim you can act on. A rule that presumes no number at all: *within 5 points of
how often two people agree with each other*. Section 14 measures that ceiling when a second person grades
the same sheet, and applies this form of the rule too.

## 1. One prepared answer, start to finish

> **In plain words.** Before measuring anything, look at one real example. Every answer is stored with
> the question, the answer, which model wrote it, what the judge said, and a **fingerprint**: a short code
> made from the lesson's text. If the lesson changes, the fingerprint stops matching and the answer is
> hidden until it's rewritten.

In [ ]:
lesson, key = "use_case_score", "learn:term"
chip = next(c for c in CHIPS[lesson] if c["key"] == key)
rec = ANSWERS[lesson][key]

print("Question:", chip["q"])
print()
print("Answer (first 400 characters):")
print(rec["a"][:400], "...")
print()
print("Written by:", rec["model"], "| status:", rec["status"])
print("Sonnet's checklist:", {k: rec["judge"][k] for k in ("correct", "answers_it", "plain", "no_giveaway", "standalone")})
print("Fingerprint still matches the lesson:", rec["fingerprint"] == tp.fingerprint(DATA, lesson, chip))

## 2. Splitting fairly: whole lessons, dealt within each phase

> **In plain words.** If you tune a judge on some answers and then test it on the *same* answers, the test
> is meaningless, like marking your own homework with the answer sheet open. So answers are split into
> practice (**dev**), exam (**test**) and locked (**holdout**) groups.
>
> Two rules from the course:
> - **Keep related things together.** All of a lesson's questions share its text, so a whole lesson goes
>   into one group ("the leakage rule").
> - **Deal fairly.** Make sure every phase lands in every group, like dealing cards to each player.

The first attempt split by a random-looking code from each lesson's id. Look what happened:

In [ ]:
plain = {lid: ("dev" if tpe._bucket(lid) < 60 else "test" if tpe._bucket(lid) < 80 else "holdout") for lid in CHIPS}
table = pd.crosstab(pd.Series([PHASE.get(l, "") for l in plain], name="phase"), pd.Series(list(plain.values()), name="split"))
table[["dev", "test", "holdout"]]

**What to look for:** the zeros. Design and Ship got **no** holdout lessons, so you could never check those
phases at the end. Pure chance missed small groups, exactly what the lesson "Dev, test and holdout sets"
warns about.

**Exercise.** Deal each phase's lessons in turn: dev, dev, dev, test, holdout, then start again. Keep a
separate count **for each phase**.

In [ ]:
PATTERN = ["dev", "dev", "dev", "test", "holdout"]  # 3 dev, 1 test, 1 holdout: 60/20/20

In [ ]:
def deal_by_phase(lessons_by_phase):
    """{phase: [lesson ids]} -> {lesson id: split}, each phase dealt in the pattern above."""
    # TODO: for each phase, give its lessons PATTERN[0], PATTERN[1], ... in turn, starting again after 5
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_every_phase_with_five_lessons_reaches_every_split():
    dealt = deal_by_phase({"Frame": ["f1", "f2", "f3", "f4", "f5"], "Ship": ["s1", "s2", "s3", "s4", "s5", "s6"]})
    assert {dealt[l] for l in ["f1", "f2", "f3", "f4", "f5"]} == {"dev", "test", "holdout"}
    assert {dealt[l] for l in ["s1", "s2", "s3", "s4", "s5"]} == {"dev", "test", "holdout"}


def test_each_phase_starts_its_own_pattern():
    dealt = deal_by_phase({"A": ["a1"], "B": ["b1", "b2", "b3", "b4"]})
    assert dealt["a1"] == "dev" and dealt["b4"] == "test"


def test_the_pattern_repeats():
    dealt = deal_by_phase({"A": [f"a{i}" for i in range(1, 8)]})
    assert [dealt[f"a{i}"] for i in range(1, 8)] == ["dev", "dev", "dev", "test", "holdout", "dev", "dev"]

The project then **freezes** the result in a file (`prepared_splits.json`), so adding lessons later never
moves old ones between groups (lesson "Splits that don't move"). Every phase now reaches every group:

In [ ]:
frozen = tpe.load_splits()
table = pd.crosstab(pd.Series([PHASE.get(l, "") for l in frozen], name="phase"), pd.Series(list(frozen.values()), name="split"))
table[["dev", "test", "holdout"]]

## 3. The sample, and what the two judges said

> **In plain words.** Judging all 1,025 practice answers takes an hour, so we judged a **sample** of 301.
> Failures are rare (Sonnet failed about 2.5%), and a purely random sample would hold only about 8 of them,
> too few to learn anything. So the sample took **every** failure, then a fair spread of the rest across
> phases and question types. That's a **stratified sample**.

Below, every judged answer becomes one row of a table: its lesson, phase, question type, length, and the
two judges' verdicts. **Sonnet** is the reference; **Haiku** is the cheaper judge we're testing.

The sample is **frozen** in `src/drill/content/prepared_sample.json`, the way the splits are: drawn once, then
kept, so every judge run grades exactly the same answers and runs compare question by question. An answer
whose lesson changed drops out and is counted; new answers join only on `--refresh-sample`, a deliberate
act the run records.

An honest note on the runs recorded in this notebook: the first version of the eval registered the sample as
an MLflow dataset and then graded *the dataset's contents*. A dataset only ever grows (new records are
upserted, never removed), so once six new lessons were placed into dev, the later runs graded 317 answers
while declaring 302, and 15 of those had not been picked by the sample rule. An adversarial review caught
it on 2026-10-18; the eval now grades the frozen sample itself and records `graded`. The recordings here
keep their 317 answers, and section 18 compares on the answers all runs share.

In [ ]:
df = pd.DataFrame(REC["answers"])
df["phase"] = df.lesson_id.map(PHASE)
df["kind"] = df.key.map(tpe.kind_of)
df["words"] = [len(ANSWERS[r.lesson_id][r.key]["a"].split()) for r in df.itertuples()]
df = df.rename(columns={"passes": "haiku", "reference_pass": "sonnet"})
df["agree"] = df.haiku == df.sonnet
print(len(df), "answers from", df.lesson_id.nunique(), "lessons")
df[["lesson_id", "kind", "phase", "words", "sonnet", "haiku"]].head(8)

## 3b. Build a representative sample yourself

> **In plain words.** A sample is **representative** when it looks like the whole set in the ways that
> matter. Pure chance usually gets close, but with small or rare groups it can miss them entirely (you saw
> that with the splits). **Stratifying** removes the luck: split the set into groups that behave
> differently, then choose from **each** group on purpose.

### Step 1: choose the groups (strata)

Stratify by anything that probably changes the result:

| Group by | Why |
|---|---|
| **Phase** (Frame, Design, Build...) | different topics; the platform track is very different from the AI path |
| **Question type** (term, stops, spot, check, interview, gotcha) | a "why not B?" answer is a different job from an interview answer |
| **Outcome** (the reference judge's pass or fail) | failures are rare (2.5%) but they're what we most need to study |

Don't over-split: with 6 types and 7 phases there are 42 little groups, some with only a few answers.
Rare groups get merged or kept whole.

### Step 2: decide how many from each group (allocation)

| Method | How many from each group | Use when |
|---|---|---|
| **Proportional** | its share of the whole set | you want the sample to mirror the set (honest overall rates) |
| **Equal** | the same number from each | you want to compare groups fairly, even small ones |
| **Over-sample the rare** | all (or extra) of a rare group, proportional for the rest | a rare group is the point (our failures) |
| **Neyman (optimal)** | more from groups that vary more | you want the most precise overall estimate for the cost |

Whenever a group isn't proportional, each answer's **weight** says how many real answers it stands for, so
overall numbers can be converted back (section 8).

### Step 3: pick inside each group without bias

Inside a group, pick in an order that has nothing to do with the answers, and that gives the **same**
sample every time you run it (so a second judge can be scored on exactly the same answers). The project
sorts by a hash of each answer's name, which works like shuffling a deck with a fixed seed.

**Exercise.** Build the sampler: take **every** row whose `rare` value is True, then fill the rest of `n`
with the other rows, **proportionally** across the groups named in `by`. Inside each group, take rows in
the order of `order_key` (given), so the result is the same every time. Round each group's share to the
nearest whole number.

In [ ]:
import hashlib


def order_key(row):
    """A fixed, meaningless order inside each group: like shuffling with a seed."""
    return hashlib.sha256(f"{row['lesson_id']}/{row['key']}".encode()).hexdigest()

In [ ]:
def stratified(rows, n, by, rare):
    """Every rare row, then the rest of n spread across the `by` groups in proportion, in order_key order.
    rows: a list of dicts; by: a list of keys to group on; rare: a function row -> True/False."""
    # TODO: rare_rows = [r for r in rows if rare(r)]; common = the others; want = n - len(rare_rows)
    #       group the common rows by tuple(r[k] for k in by); for each group (in sorted order), sort it by
    #       order_key and take round(want * len(group) / len(common)) rows; return rare_rows + the taken rows
    raise NotImplementedError

In [ ]:
%%ipytest -qq

ROWS = [{"lesson_id": f"l{i}", "key": "k", "phase": "A" if i < 60 else "B", "fail": i % 20 == 0} for i in range(100)]


def test_keeps_every_rare_row():
    picked = stratified(ROWS, 25, ["phase"], lambda r: r["fail"])
    assert sum(r["fail"] for r in picked) == 5


def test_spreads_the_rest_in_proportion():
    picked = [r for r in stratified(ROWS, 25, ["phase"], lambda r: r["fail"]) if not r["fail"]]
    assert Counter(r["phase"] for r in picked) == {"A": 12, "B": 8}  # 57 and 38 common rows: 60% and 40% of 20


def test_the_same_every_time():
    assert stratified(ROWS, 25, ["phase"], lambda r: r["fail"]) == stratified(ROWS, 25, ["phase"], lambda r: r["fail"])

Now build a 300-answer sample of the whole **dev** split, grouped by phase and question type, keeping every
failure, and compare it with the project's own sample.

In [ ]:
DEV = []
for lid, items in CHIPS.items():
    if frozen.get(lid) != "dev":
        continue
    for c in items:
        r = ANSWERS[lid][c["key"]]
        if r.get("q") == c["q"] and r.get("a"):
            DEV.append({"lesson_id": lid, "key": c["key"], "phase": PHASE.get(lid, ""), "kind": tpe.kind_of(c["key"]),
                        "fail": not r["judge"]["pass"]})

mine = stratified(DEV, 300, ["phase", "kind"], lambda r: r["fail"])
theirs = {(a["lesson_id"], a["key"]) for a in REC["answers"]}
print(f"dev has {len(DEV)} answers ({sum(r['fail'] for r in DEV)} failed); your sample: {len(mine)} "
      f"({sum(r['fail'] for r in mine)} failed); the same answers as the project's: {sum((r['lesson_id'], r['key']) in theirs for r in mine)} of {len(mine)}")

### Step 4: check the sample really is representative

Compare the **share** of each group in the sample with its share in the whole set. Failures are
over-sampled on purpose, so compare only the **passed** answers, the part meant to be proportional.

A **chi-square goodness-of-fit test** asks: if we'd sampled fairly, how surprising would differences this
big be? A large p-value (above 0.05) means "no sign the mix is off". It's the same idea as McNemar: how
surprising is what we see, if nothing's wrong?

In [ ]:
from scipy.stats import chisquare

common_all = [r for r in DEV if not r["fail"]]
common_mine = [r for r in mine if not r["fail"]]
for by in ("phase", "kind"):
    pop = Counter(r[by] for r in common_all)
    smp = Counter(r[by] for r in common_mine)
    groups = sorted(pop)
    table = pd.DataFrame({"share_of_dev": [pop[g] / len(common_all) for g in groups],
                          "share_of_sample": [smp[g] / len(common_mine) for g in groups]}, index=groups).round(3)
    table["difference"] = (table.share_of_sample - table.share_of_dev).round(3)
    expected = [pop[g] / len(common_all) * len(common_mine) for g in groups]
    stat, p = chisquare([smp[g] for g in groups], expected)
    print(f"by {by}: largest difference {table.difference.abs().max():.1%}, chi-square p = {p:.2f}")
    display(table)

**What to look for.** Small differences (a percentage point or so, from rounding) and a large p-value: the
sample mirrors dev's mix. If one group were badly off, you'd see it in the `difference` column first.

**Two cautions a professional would add:**

- **Representative of what?** This sample mirrors dev's **questions**. It says nothing about which questions
  learners actually **ask** most. For that, weight by real traffic from the `ask_lesson` traces.
- **Representative only on what you grouped by.** Stratifying by phase and type can't balance something you
  didn't group on, like answer length. Check the ones you care about separately (section 10 does length).

## 4. The confusion matrix: four kinds of outcome

> **In plain words.** For a checker whose job is to catch bad answers, every answer lands in one of four
> boxes:
>
> | | Reference says FAIL | Reference says PASS |
> |---|---|---|
> | **Judge says FAIL** | caught it (true positive) | false alarm |
> | **Judge says PASS** | missed it | correctly let through (true negative) |
>
> Here "positive" means "flagged as a problem". From the boxes:
> - **Recall**: of the truly bad answers, how many did the judge catch? (Missing bad answers is the danger.)
> - **Precision**: of the answers the judge flagged, how many were really bad? (Low means lots of false alarms.)
> - **Specificity**: of the good answers, how many did it let through?

**Exercise.** `judge` and `reference` are lists of True (pass) or False (fail). Count the four boxes, then
work out recall, precision and specificity. If a rate has nothing to divide by, use `None`.

In [ ]:
def confusion(judge, reference):
    """The four boxes and three rates, where 'positive' means the answer FAILED (a problem was flagged)."""
    # TODO: caught = judge failed and reference failed; false_alarm = judge failed, reference passed;
    #       missed = judge passed, reference failed; let_through = both passed.
    #       recall = caught / (caught + missed); precision = caught / (caught + false_alarm);
    #       specificity = let_through / (let_through + false_alarm)
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_the_four_boxes_and_rates():
    judge = [False, False, True, False, True, True]
    ref = [False, True, False, False, True, True]
    out = confusion(judge, ref)
    assert (out["caught"], out["false_alarm"], out["missed"], out["let_through"]) == (2, 1, 1, 2)
    assert out["recall"] == 2 / 3 and out["precision"] == 2 / 3 and out["specificity"] == 2 / 3


def test_nothing_to_divide_by():
    assert confusion([True], [True])["recall"] is None

In [ ]:
cm = confusion(list(df.haiku), list(df.sonnet))
print(pd.DataFrame([[cm["caught"], cm["false_alarm"]], [cm["missed"], cm["let_through"]]],
                   index=["Haiku FAIL", "Haiku PASS"], columns=["Sonnet FAIL", "Sonnet PASS"]))
print(f"\nrecall {cm['recall']:.0%}   precision {cm['precision']:.0%}   specificity {cm['specificity']:.0%}")

**What to look for.** Haiku catches most of what Sonnet failed (recall around 85%), but its **precision
is tiny**: almost everything it flags, Sonnet passed. As a daily monitor it would cry wolf constantly.
Remember though: Sonnet isn't the truth. Section 11 is where a person decides.

## 5. How sure is a rate? Intervals, the simple way

> **In plain words.** "36% of 275" is an estimate. With a different sample you'd get a slightly different
> number. A **confidence interval** is the range the true rate probably sits in. The course's rule of thumb:
> the margin is about **100 divided by the square root of the sample size**, in percentage points (so 300
> answers: about plus or minus 6 points).
>
> The **Wilson interval** (lesson "How sure is that pass rate?") does it properly, and behaves well even for
> small groups or rates near 0% or 100%. In words: start from the observed rate, pull it a little toward the
> middle (more when the sample is small), and spread out by an amount that shrinks as the sample grows.

**Exercise.** Return `(rate, low, high)`. With `n = 0`, return `(0.0, 0.0, 0.0)`.

In [ ]:
def wilson(passed, n, z=1.96):
    """A rate and its Wilson 95% interval."""
    # TODO: p = passed / n; centre = (p + z*z/(2n)) / (1 + z*z/n);
    #       half = z * sqrt(p(1-p)/n + z*z/(4n*n)) / (1 + z*z/n); return p, centre - half, centre + half
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_ninety_out_of_a_hundred():
    rate, low, high = wilson(90, 100)
    assert rate == 0.9 and round(low, 3) == 0.826 and round(high, 3) == 0.945


def test_nothing_measured():
    assert wilson(0, 0) == (0.0, 0.0, 0.0)


def test_it_never_leaves_zero_to_one():
    assert wilson(10, 10)[2] <= 1.0 and wilson(0, 10)[1] >= 0.0

## 6. Answers aren't independent: the lesson-level bootstrap

> **In plain words.** Wilson assumes every answer is a separate, unrelated coin flip. They aren't: answers
> about the **same lesson** tend to pass or fail together (same text, same tricky terms). 301 answers from
> about 100 lessons carry less information than 301 unrelated answers, so Wilson's interval is **too
> narrow**: it makes us sound surer than we are.
>
> The fix is a **bootstrap that resamples whole lessons**: build a new sample by picking lessons at random
> (some twice, some not at all), keeping all of each picked lesson's answers, and recompute the number.
> Do that a thousand times; the middle 95% of the results is the interval. It's a "cluster bootstrap".

This one is written for you; read it line by line.

In [ ]:
def cluster_bootstrap(rows, stat, n_boot=1000, seed=7):
    """95% interval for stat(rows), resampling whole lessons. rows: a DataFrame with a lesson_id column."""
    rng = random.Random(seed)                              # a fixed seed: the same result every time you run it
    by_lesson = {lid: grp for lid, grp in rows.groupby("lesson_id")}
    lessons = list(by_lesson)
    results = []
    for _ in range(n_boot):
        picked = [rng.choice(lessons) for _ in lessons]    # as many lessons as before, chosen with replacement
        sample = pd.concat([by_lesson[l] for l in picked])
        value = stat(sample)
        if value is not None:
            results.append(value)
    results.sort()
    return results[int(0.025 * len(results))], results[int(0.975 * len(results)) - 1]


keep = lambda d: (d[d.sonnet].haiku).mean()                # of Sonnet's passes, the share Haiku passed too
catch = lambda d: (~d[~d.sonnet].haiku).mean() if (~d.sonnet).any() else None

for name, fn, hit, n in (("keep", keep, int(df[df.sonnet].haiku.sum()), int(df.sonnet.sum())),
                         ("catch", catch, int((~df[~df.sonnet].haiku).sum()), int((~df.sonnet).sum()))):
    _, wl, wh = tpe.wilson(hit, n)
    bl, bh = cluster_bootstrap(df, fn)
    print(f"{name:5}: {hit / n:.0%}   Wilson {wl:.0%}-{wh:.0%} (width {wh - wl:.0%})   lesson bootstrap {bl:.0%}-{bh:.0%} (width {bh - bl:.0%})")

**What to look for.** The lesson-level intervals are **wider**. That's the honest size of our uncertainty.
A professional reports these, not the narrow ones.

**The design effect** puts a number on it: how many times wider (squared) the honest interval is. A design
effect of 2 means our 301 answers are worth about 150 independent ones.

In [ ]:
_, wl, wh = tpe.wilson(int(df[df.sonnet].haiku.sum()), int(df.sonnet.sum()))
bl, bh = cluster_bootstrap(df, keep)
effect = ((bh - bl) / (wh - wl)) ** 2
print(f"design effect about {effect:.1f}: the {int(df.sonnet.sum())} passed answers are worth about {df.sonnet.sum() / effect:.0f} independent ones")

## 7. Do two judges agree? Kappa, and why it misbehaves here

> **In plain words.** If almost every answer passes, two judges will agree most of the time just by both
> saying "pass". **Cohen's kappa** asks: how much better than luck? 1 is perfect, 0 is no better than
> luck.
>
> - **Observed**: how often they agree.
> - **Chance**: how often they'd agree by luck = (A's pass share x B's pass share) + (A's fail share x B's fail share).
> - **Kappa** = (observed - chance) / (1 - chance). If chance is 1, kappa is undefined: return `None`.

**Exercise.**

In [ ]:
def kappa(a, b):
    """Cohen's kappa between two judges' pass/fail verdicts."""
    # TODO: observed agreement, chance agreement, then (observed - chance) / (1 - chance); None if chance is 1
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_ninety_six_percent_agreement_is_not_as_good_as_it_sounds():
    a = [True] * 96 + [False] * 4
    b = [True] * 98 + [False] * 2
    assert round(kappa(a, b), 2) == 0.66


def test_perfect_agreement():
    assert kappa([True, False, True, False], [True, False, True, False]) == 1.0


def test_no_variation_means_no_answer():
    assert kappa([True] * 5, [True] * 5) is None

> **The kappa paradox.** When one outcome is very rare (here, failing), kappa can come out **low even when
> two judges mostly agree**, because the "chance" part becomes huge. **Gwet's AC1** fixes that by
> estimating chance differently: if `q` is the average pass share of the two judges, chance =
> 2 x q x (1 - q). Then AC1 = (observed - chance) / (1 - chance), the same shape as kappa.

**Exercise.** Write AC1, then compare the two on the example from the test above.

In [ ]:
def ac1(a, b):
    """Gwet's AC1 between two judges' pass/fail verdicts."""
    # TODO: observed agreement; q = (A's pass share + B's pass share) / 2; chance = 2q(1-q);
    #       (observed - chance) / (1 - chance); None if chance is 1
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_ac1_stays_high_when_judges_mostly_agree_on_rare_failures():
    a = [True] * 96 + [False] * 4
    b = [True] * 98 + [False] * 2
    assert ac1(a, b) > 0.9 > kappa(a, b)


def test_ac1_perfect():
    assert ac1([True, False, True], [True, False, True]) == 1.0

In [ ]:
a, b = list(df.haiku), list(df.sonnet)
print(f"observed agreement {sum(x == y for x, y in zip(a, b)) / len(a):.0%}")
print(f"kappa {kappa(a, b):.2f}   AC1 {ac1(a, b):.2f}")
kl, kh = cluster_bootstrap(df, lambda d: kappa(list(d.haiku), list(d.sonnet)))
al, ah = cluster_bootstrap(df, lambda d: ac1(list(d.haiku), list(d.sonnet)))
print(f"lesson-bootstrap intervals: kappa {kl:.2f} to {kh:.2f}, AC1 {al:.2f} to {ah:.2f}")

**What to look for.** Here both are low, and AC1 doesn't rescue Haiku: the judges genuinely disagree a lot,
it isn't just the paradox. Note too that this sample over-represents failures on purpose, so these numbers
describe the sample. Section 8 fixes that.

## 8. Undoing the over-sampling: weights

> **In plain words.** We put *all* 26 failures in the sample but only about 1 in 4 of the passes. So each
> sampled pass "stands for" about 4 real ones, and each failure stands for 1. Giving every row that
> **weight** turns sample numbers back into estimates for the whole dev set.

In [ ]:
share_failed = REC["metrics"]["population_failed_share"]  # the share of ALL dev answers Sonnet failed (recorded by the run)
w_fail = share_failed / (~df.sonnet).mean()           # how many real answers each sampled failure stands for (relative)
w_pass = (1 - share_failed) / df.sonnet.mean()
df["weight"] = df.sonnet.map({False: w_fail, True: w_pass})


def weighted_agreement(d):
    return (d.agree * d.weight).sum() / d.weight.sum()


lo, hi = cluster_bootstrap(df, weighted_agreement)
print(f"failures are {share_failed:.1%} of dev but {(~df.sonnet).mean():.1%} of the sample")
print(f"agreement in the sample {df.agree.mean():.0%}; weighted to dev's real mix {weighted_agreement(df):.0%} (lesson bootstrap {lo:.0%}-{hi:.0%})")

## 9. Where do they disagree, and why?

> **In plain words.** One overall number hides *where* things go wrong. Split by group. But testing many
> groups creates a new trap: test 13 groups and one will look "significant" by luck alone. So adjust the
> p-values for how many you tested.

First, agreement by phase and by question type, each with a p-value for "is this group's keep rate
different from the rest?"

In [ ]:
def two_rate_p(hit1, n1, hit2, n2):
    """A simple two-sided test that two rates differ (normal approximation): small p means probably real."""
    if not n1 or not n2:
        return 1.0
    p = (hit1 + hit2) / (n1 + n2)
    se = math.sqrt(p * (1 - p) * (1 / n1 + 1 / n2)) or 1e-9
    z = abs(hit1 / n1 - hit2 / n2) / se
    return math.erfc(z / math.sqrt(2))


rows = []
passed = df[df.sonnet]
for by in ("phase", "kind"):
    for group, g in passed.groupby(by):
        rest = passed[passed[by] != group]
        rows.append({"by": by, "group": group, "n": len(g), "keep_rate": round(g.haiku.mean(), 2),
                     "p": two_rate_p(int(g.haiku.sum()), len(g), int(rest.haiku.sum()), len(rest))})
groups = pd.DataFrame(rows)
groups

**Exercise: the Holm adjustment.** Sort the p-values from smallest to largest. Multiply the smallest by the
number of tests, the next by one fewer, and so on. Each adjusted value must be at least as big as the one
before it, and never more than 1. Return them **in the original order**.

In [ ]:
def holm(pvalues):
    """Holm-adjusted p-values, in the same order as given."""
    # TODO: order = indexes sorted by p; walk them, adjusted = max(previous, min(1, (m - rank) * p))
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_holm_on_three():
    assert [round(x, 3) for x in holm([0.01, 0.04, 0.03])] == [0.03, 0.06, 0.06]


def test_never_above_one():
    assert holm([0.5, 0.9]) == [1.0, 1.0]

In [ ]:
groups["p_adjusted"] = holm(list(groups.p))
groups.sort_values("p_adjusted").round(3)

**What to look for.** Which groups stay below 0.05 *after* adjusting. Those are real differences worth
acting on; the rest may be luck.

Now **why**. Haiku writes a reason for every fail. Sorting reasons into categories by keywords is rough,
and a person reading them would do better, but it shows the shape:

In [ ]:
CATEGORIES = {
    "unexplained jargon": ("jargon", "unexplained", "never explain", "without explain", "not explained", "acronym", "term"),
    "claims the lesson doesn't make": ("lesson never", "lesson does not", "lesson doesn't", "not in the lesson", "overstat", "overclaim", "contradict", "goes beyond"),
    "too long": ("words", "too long", "sentences", "length"),
    "gives something away": ("exercise", "solution", "gives away", "answer away"),
}


def category(note):
    note = (note or "").lower()
    return next((name for name, words in CATEGORIES.items() if any(w in note for w in words)), "other")


flagged = df[~df.haiku & df.sonnet].copy()
flagged["category"] = flagged.notes.map(category)
print(flagged.category.value_counts().to_string())
for name in flagged.category.unique():
    print(f"\n{name}, for example:")
    for note in flagged[flagged.category == name].notes.head(2):
        print("  -", note[:150])

## 10. Is the judge biased by length?

> **In plain words.** AI judges are known to be swayed by things that shouldn't matter, length most of all.
> If Haiku fails long answers more often, part of its strictness might just be about length. Split answers
> into four equal-sized length groups and compare pass rates.

In [ ]:
df["length_group"] = pd.qcut(df.words, 4, labels=["shortest", "short", "long", "longest"])
by_length = df.groupby("length_group", observed=True).agg(answers=("words", "size"), median_words=("words", "median"),
                                                         sonnet_passes=("sonnet", "mean"), haiku_passes=("haiku", "mean")).round(2)
print(by_length.to_string())
print(f"\ncorrelation of length with passing: Haiku {df.words.corr(df.haiku.astype(float)):+.2f}, Sonnet {df.words.corr(df.sonnet.astype(float)):+.2f}")

**What to look for.** If Haiku's pass rate falls clearly as answers get longer while Sonnet's doesn't,
that's a length bias. A correlation near 0 means length isn't driving it.

## 11. Is the judge consistent? Run it twice

> **In plain words.** A thermometer that reads differently every time you use it is useless, however
> accurate it is on average. Judges are the same: run Haiku twice on the **same** answers and see how often
> it changes its mind. If it flips often, every other number in this notebook is noisy.

In [ ]:
rerun_path = RECORDINGS / "dev_sample_haiku_rerun.json"
if rerun_path.exists():
    rerun = {(a["lesson_id"], a["key"]): a["passes"] for a in json.loads(rerun_path.read_text())["answers"]}
    first = list(df.haiku)
    second = [rerun.get((r.lesson_id, r.key)) for r in df.itertuples()]
    both = [(x, y) for x, y in zip(first, second) if y is not None]
    same = sum(x == y for x, y in both)
    print(f"Haiku gave the same verdict twice on {same} of {len(both)} answers ({same / len(both):.0%})")
    print(f"kappa between its two runs {kappa([x for x, _ in both], [y for _, y in both]):.2f}, AC1 {ac1([x for x, _ in both], [y for _, y in both]):.2f}")
else:
    print("The second run's recording isn't here yet: run `uv run drill tutor prepared-eval --profile DEFAULT --split dev --sample 300` again.")

## 12. Comparing two versions: McNemar, and how many answers you need

> **In plain words.** When you change the judge's instructions and run it again on the **same** answers,
> only the answers where the two runs **disagree** tell you which is better. If there's no real
> difference, each disagreement is a coin flip. McNemar's test asks how surprising your split of
> disagreements would be under coin flips (lesson "Real win or luck").

**Exercise.** The chance of `min(b, c)` or fewer heads in `b + c` fair flips, doubled, capped at 1.
`math.comb(n, k)` counts the ways to get `k` heads in `n` flips; there are `2 ** n` outcomes in all.

In [ ]:
def mcnemar(b, c):
    """Exact two-sided McNemar p-value from the disagreements alone."""
    # TODO: n = b + c; add up math.comb(n, k) for k from 0 to min(b, c); divide by 2 ** n; double; cap at 1
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_twelve_against_eight_is_too_close_to_call():
    assert round(mcnemar(12, 8), 3) == 0.503


def test_fifteen_against_two_is_real():
    assert mcnemar(15, 2) < 0.01


def test_it_does_not_matter_which_side_is_which():
    assert mcnemar(3, 9) == mcnemar(9, 3)


def test_no_disagreements():
    assert mcnemar(0, 0) == 1.0

> **How many answers do you need?** Before running an experiment, check it can actually detect the
> difference you care about. Otherwise a "no difference" result just means "too small to tell". This is
> called **power**: the chance the test spots a real difference of a given size.
>
> The cell below simulates it: imagine the new prompt really fixes some share of the old run's mistakes and
> breaks a smaller share of its right answers, run McNemar many times on made-up data like that, and count how
> often it says "real" (p under 0.05).

In [ ]:
def power(n_answers, fixes, breaks, trials=2000, seed=7):
    """Chance McNemar spots the difference when the new version fixes `fixes` and breaks `breaks` (shares of answers)."""
    rng = random.Random(seed)
    hits = 0
    for _ in range(trials):
        b = sum(rng.random() < fixes for _ in range(n_answers))
        c = sum(rng.random() < breaks for _ in range(n_answers))
        hits += mcnemar(b, c) < 0.05
    return hits / trials


for n in (100, 300, 1000):
    print(f"{n:5} answers: power {power(n, fixes=0.06, breaks=0.02):.0%} to detect fixing 6% and breaking 2%")

**What to look for.** Around 80% is the usual target. If 300 answers give much less, the experiment can't
answer its question: use more answers, or only look for bigger effects.

The project has the same McNemar function: `uv run drill tutor prepared-compare RUN_A RUN_B` runs it on two
real runs, answer by answer.

## 13. Is it worth the money? Cost against quality

> **In plain words.** A cheaper judge is only a bargain if it's good enough. Put cost and quality side by
> side (the course's lesson on choosing a model by quality, cost and speed).

In [ ]:
m = REC["metrics"]
# from the run's own token counts, at today's list price (the run itself logged a stand-in price)
haiku_per_answer = tp.cost(REC["params"]["judge_model"], m["judge_input_tokens"], m["judge_output_tokens"]) / m["judge_calls"]
# Sonnet's judging during drafting, from the workspace's usage table on 2026-10-17: 2,094 calls
sonnet_per_answer = tp.cost("databricks-claude-sonnet-5-5", 3_429_008, 2_212_630) / 2094

options = pd.DataFrame([
    {"judge": "Sonnet 5.5 (reference)", "cents_per_answer": 100 * sonnet_per_answer, "agrees_with_reference": 1.0,
     "precision_vs_reference": 1.0, "note": "the reference: agreement with itself is 1 by definition"},
    {"judge": "Haiku 5.5", "cents_per_answer": 100 * haiku_per_answer, "agrees_with_reference": round(weighted_agreement(df), 2),
     "precision_vs_reference": round(cm["precision"], 2), "note": f"{int(m['judge_rate_limited'])} calls rate-limited"},
])
options["cost_for_1000_answers_usd"] = (options.cents_per_answer * 10).round(2)
options.round(2)

**What to look for.** Haiku is cheaper per answer, but at this agreement and precision it would bury real
problems under false alarms. The honest comparison, though, needs a person's verdicts as the truth,
which is the next section.

## 14. Your turn: build the golden set (the ground truth)

> **In plain words.** Everything so far compared one AI with another. To know which is right, a **person**
> grades a sample: that's the **golden set**. A professional would also have **two people** grade the same
> answers separately and measure how much *they* agree. That's the ceiling: no judge can be expected to
> beat human-to-human agreement.

The cell below writes a spreadsheet of about 60 answers to grade: every disagreement Haiku and Sonnet had
on failures, plus a spread of the rest. Open it, fill the `your_verdict` column with `pass` or `fail` (and
`second_person` if someone else grades too), save it, then run the cell after.

In [ ]:
golden_path = Path("golden_to_label.csv")
if not golden_path.exists():
    on_failures = df[~df.sonnet | ~df.haiku]
    pick = pd.concat([on_failures[~on_failures.sonnet], on_failures[on_failures.sonnet].sample(n=min(25, int(on_failures.sonnet.sum())), random_state=7),
                      df[df.sonnet & df.haiku].sample(n=10, random_state=7)]).drop_duplicates(["lesson_id", "key"])
    sheet = pick[["lesson_id", "key"]].copy()
    sheet["question"] = [ANSWERS[r.lesson_id][r.key]["q"] for r in pick.itertuples()]
    sheet["answer"] = [ANSWERS[r.lesson_id][r.key]["a"] for r in pick.itertuples()]
    sheet["your_verdict"] = ""
    sheet["second_person"] = ""
    sheet.sample(frac=1, random_state=7).to_csv(golden_path, index=False)  # shuffled, and judges' verdicts hidden: no peeking
    print(f"wrote {len(sheet)} answers to {golden_path.resolve()}")
else:
    print(f"{golden_path} already exists: fill it in, then run the next cell")

In [ ]:
labels = pd.read_csv(golden_path).fillna("")
labels = labels[labels.your_verdict.str.strip().str.lower().isin(["pass", "fail"])]
if len(labels):
    merged = labels.merge(df[["lesson_id", "key", "haiku", "sonnet"]], on=["lesson_id", "key"])
    you = merged.your_verdict.str.strip().str.lower() == "pass"
    for judge in ("haiku", "sonnet"):
        c = confusion(list(merged[judge]), list(you))
        print(f"{judge:6} vs you: recall {c['recall']}, precision {c['precision']}, specificity {c['specificity']}, "
              f"AC1 {ac1(list(merged[judge]), list(you)):.2f}")
    two = merged[merged.second_person.str.strip().str.lower().isin(["pass", "fail"])]
    if len(two):
        other = two.second_person.str.strip().str.lower() == "pass"
        mine = two.your_verdict.str.strip().str.lower() == "pass"
        print(f"you vs the second person on {len(two)}: kappa {kappa(list(mine), list(other))}, AC1 {ac1(list(mine), list(other)):.2f}  <- the ceiling")
        ceiling = float((mine == other).mean())                          # how often two people agree: no judge beats this for long
        print(f"two people agree on {ceiling:.0%} of answers. The rule in the ceiling's terms: a judge must agree with you on at least {ceiling - 0.05:.0%}")
        for judge in ("haiku", "sonnet"):
            with_you = float((merged[judge] == you).mean())
            print(f"  {judge:6} agrees with you on {with_you:.0%}: {'within 5 points of the ceiling' if with_you >= ceiling - 0.05 else 'below it'}")
    print("\nWhen you're happy: record them with `uv run drill tutor prepared --check lesson_id/key` or `--reject lesson_id/key`")
else:
    print("No labels yet: fill in your_verdict in golden_to_label.csv")

## 15. Track it in MLflow, hands-on

> **In plain words.** An experiment you can't find again, or can't reproduce, didn't really happen.
> MLflow is the lab notebook: every **run** records *what you did* (parameters), *what you got* (metrics),
> *labels to find it by* (tags), and *the evidence* (artifacts: tables, charts, the plan). **Datasets** are
> versioned, so a run always points at the exact data it used. An **eval run** is a run made by
> `mlflow.genai.evaluate`: one trace per answer, with the scores attached.

| Record | What goes in it here | Why |
|---|---|---|
| **Experiment** | one per question: "can a cheaper judge stand in?" | groups the runs you'll compare |
| **Dataset** | the 301 answers, with the reference verdicts as expectations, version-tagged | the run names the exact data (its digest) |
| **Run name** | `judge-<model>-<prompt version>-<split>` | readable at a glance, sortable |
| **Params** | judge model, prompt version, split, sample rule, seed, decision rule | what you'd need to rerun it |
| **Tags** | split, purpose (exploration / final report), look number, decision | find runs; audit how often test was used |
| **Metrics** | recall, precision, specificity, kappa, AC1, with interval ends | the results, comparable across runs |
| **Artifacts** | the plan, the confusion matrix, the group table, a chart | the evidence behind the numbers |
| **Traces** | one per answer, with the judge's verdict and reason attached | drill into any single answer |

This uses a **throwaway** MLflow store in a temporary folder, and replays the recorded verdicts, so it
costs nothing and needs no server. Set `USE_LOCAL_SERVER = True` to send it to your MLflow at
`http://localhost:5050` instead, and see it in the UI.

In [ ]:
import os
import tempfile

os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")   # avoids a known threading hang inside evaluate
os.environ.setdefault("MLFLOW_GENAI_EVAL_MAX_WORKERS", "4")
import mlflow
import mlflow.genai.judges  # noqa: F401  (loaded up front for the same reason)
import mlflow.metrics.genai  # noqa: F401
from mlflow.entities import AssessmentSource, Feedback
from mlflow.genai.scorers import scorer

USE_LOCAL_SERVER = False
if USE_LOCAL_SERVER:
    mlflow.set_tracking_uri("http://localhost:5050")
else:
    STORE = tempfile.mkdtemp(prefix="tutor-mlflow-")
    mlflow.set_tracking_uri(f"sqlite:///{STORE}/mlflow.db")
experiment = mlflow.set_experiment("tutor-judge-learning")
print("tracking to", mlflow.get_tracking_uri(), "| experiment", experiment.name, experiment.experiment_id)

**Step 1: register the dataset.** Each record has **inputs** (what the system under test receives) and
**expectations** (what a good result looks like). Here the expectation is the reference judge's verdict.
Tags on the dataset say which version it is and how it was built.

In [ ]:
records = [{"inputs": {"lesson_id": a["lesson_id"], "key": a["key"], "phase": PHASE.get(a["lesson_id"], ""), "kind": tpe.kind_of(a["key"])},
            "expectations": {"reference_pass": a["reference_pass"], "recorded_judge_pass": a["passes"], "recorded_reason": a["notes"]}}
           for a in REC["answers"]]
try:
    ds = mlflow.genai.get_dataset(name="tutor_dev_sample")
except Exception:
    ds = mlflow.genai.create_dataset(name="tutor_dev_sample", experiment_id=experiment.experiment_id,
                                     tags={"version": "dev_sample-v1", "split": "dev", "rule": "every reference failure + passes spread by phase and kind"})
ds.merge_records(records)
print(f"dataset {ds.name}: {len(ds.to_df())} records, digest {ds.digest}  <- the run will record this")

**Step 2: define what's being evaluated, and the scorers.**

- `answer_for` is the **system under test**: given a question, it returns the prepared answer (the same
  thing a learner sees). `@mlflow.trace` makes each call a trace.
- `replayed_judge` is a **scorer**: it returns the judge's verdict as a `Feedback` with a reason. Here it
  replays the recorded Haiku verdict; live, it would call the model (that's what `drill tutor prepared-eval` does).
- `word_count` is a **code scorer**: plain Python, no AI, free.

In [ ]:
@mlflow.trace(name="tutor.prepared_answer", span_type="CHAIN")
def answer_for(lesson_id, key, phase="", kind=""):
    mlflow.update_current_trace(tags={"lesson_id": lesson_id, "question_kind": kind, "phase": phase, "split": "dev"})
    return ANSWERS[lesson_id][key]["a"]


@scorer(name="judge")
def replayed_judge(outputs, expectations):
    return Feedback(name="passes", value=bool(expectations["recorded_judge_pass"]), rationale=expectations["recorded_reason"] or None,
                    source=AssessmentSource(source_type="LLM_JUDGE", source_id=REC["params"]["judge_model"]))


@scorer(name="word_count")
def word_count(outputs):
    return len(outputs.split())

**Step 3: the eval run.** Everything inside `with mlflow.start_run(...)` belongs to one run: the params and
tags first, then `evaluate` (which writes the traces and scores), then our own metrics and artifacts.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

run_name = f"judge {REC['params']['judge_model'].removeprefix('databricks-')} · databricks inline · replay · dev-sample-{len(REC['answers'])}"
with mlflow.start_run(run_name=run_name) as run:
    mlflow.set_tags({"split": "dev", "purpose": "exploration", "look_kind": "dev",
                     "mlflow.note.content": (
                         "**Question:** can Haiku, a cheaper judge, stand in for Sonnet when checking the tutor's prepared answers?\n\n"
                         f"**What ran:** a replay of the recorded {REC['params']['judge_model']} verdicts (MLflow run {REC.get('run_id', '?')}) "
                         "on the stratified dev sample; Sonnet's verdicts from the drafting run are the reference.\n\n"
                         "**How it's measured:** recall and specificity with lesson-level bootstrap intervals, precision, kappa, AC1, "
                         f"agreement re-weighted to dev's mix. Rule: {PLAN['decision_rule']}\n\n"
                         "**Reproduce:** `uv run drill tutor prepared-eval --profile DEFAULT --split dev --sample 300` for the real run; "
                         "this replay is section 15 of notebooks/learning/evaluating_the_tutor.ipynb.")})
    mlflow.log_params({"judge_model": REC["params"]["judge_model"], "judge_prompt": REC["params"]["judge_prompt"],
                       "reference_judge": REC["params"]["reference_judge"], "split": "dev", "dataset": ds.name,
                       "dataset_digest": ds.digest, "sample_rule": "all reference failures + passes by phase and kind",
                       "bootstrap": "1000 resamples of whole lessons, seed 7", "decision_rule": PLAN["decision_rule"]})
    result = mlflow.genai.evaluate(data=records, predict_fn=answer_for, scorers=[replayed_judge, word_count])

    # our own metrics: each with its interval ends, so runs can be compared honestly
    cm = confusion(list(df.haiku), list(df.sonnet))
    recall_lo, recall_hi = cluster_bootstrap(df, lambda d: confusion(list(d.haiku), list(d.sonnet))["recall"])
    spec_lo, spec_hi = cluster_bootstrap(df, lambda d: confusion(list(d.haiku), list(d.sonnet))["specificity"])
    mlflow.log_metrics({"recall": cm["recall"], "recall_ci_low": recall_lo, "recall_ci_high": recall_hi,
                        "precision": cm["precision"], "specificity": cm["specificity"], "specificity_ci_low": spec_lo,
                        "specificity_ci_high": spec_hi, "kappa": kappa(list(df.haiku), list(df.sonnet)),
                        "ac1": ac1(list(df.haiku), list(df.sonnet)), "weighted_agreement": weighted_agreement(df),
                        "cents_per_answer": 100 * haiku_per_answer})
    passed_rule = cm["recall"] >= 0.8 and cm["specificity"] >= 0.8 and recall_lo >= 0.7 and spec_lo >= 0.7
    mlflow.set_tag("decision", "trust for monitoring" if passed_rule else "do not trust yet")

    # artifacts: the evidence
    mlflow.log_dict(PLAN, "plan.json")
    mlflow.log_table({"judge": ["FAIL", "PASS"], "reference FAIL": [cm["caught"], cm["missed"]],
                      "reference PASS": [cm["false_alarm"], cm["let_through"]]}, "confusion_matrix.json")
    mlflow.log_table(groups.round(4).to_dict(orient="list"), "groups.json")
    by_phase = df[df.sonnet].groupby("phase").haiku.agg(["mean", "size"])
    fig, ax = plt.subplots(figsize=(7, 3))
    for i, (ph, row) in enumerate(by_phase.iterrows()):
        _, lo, hi = tpe.wilson(int(round(row["mean"] * row["size"])), int(row["size"]))
        ax.errorbar(row["mean"], i, xerr=[[row["mean"] - lo], [hi - row["mean"]]], fmt="o", capsize=3)
    ax.set_yticks(range(len(by_phase)), by_phase.index)
    ax.set_xlim(0, 1)
    ax.set_xlabel("keep rate (Haiku passes what Sonnet passed), 95% interval")
    fig.tight_layout()
    mlflow.log_figure(fig, "keep_rate_by_phase.png")
    plt.close(fig)

print("run", run.info.run_id[:8], "| decision:", "trust" if passed_rule else "do not trust yet")

**Step 4: read it back**, the way you'd audit someone else's run. If you can't answer "what data, what
settings, what result, what evidence?" from MLflow alone, something wasn't logged.

In [ ]:
r = mlflow.get_run(run.info.run_id)
print("name:   ", r.info.run_name)
print("tags:   ", {k: v for k, v in r.data.tags.items() if not k.startswith("mlflow.")})
print("params: ", {k: r.data.params[k] for k in ("judge_model", "judge_prompt", "dataset", "dataset_digest", "split")})
print("metrics:", {k: round(v, 3) for k, v in r.data.metrics.items() if k in ("recall", "recall_ci_low", "specificity", "kappa", "ac1")})
print("artifacts:", [a.path for a in mlflow.MlflowClient().list_artifacts(run.info.run_id)])
t = mlflow.search_traces(run_id=run.info.run_id, max_results=1, return_type="list")[0]
print("one trace:", t.info.tags.get("lesson_id"), "| spans:", [s.name for s in t.data.spans],
      "| verdicts:", [(a.name, a.value) for a in t.info.assessments][:3])

## 16. Using test and holdout properly

> **In plain words.** Every time you look at a set's results and change something because of them, that
> set teaches you a little, and stops being a fair exam. So each set has a rule:
>
> | Set | When you may look | What it's for |
> |---|---|---|
> | **Dev** | as often as you like | trying prompts, judges, thresholds |
> | **Test** | once per candidate you're about to ship | the number you report |
> | **Holdout** | rarely, for a big decision, with a written reason | a last, untouched check; if you ever tune after looking, it becomes dev and you lock a new holdout |
>
> The way to keep yourself honest is to **record every look** as a tagged run, so anyone can count them.

**Exercise.** Write the rule. `looks_so_far` is how many times this split was already looked at for this
candidate. Return `True` if looking is allowed:

- dev: always;
- test: only if it hasn't been looked at yet for this candidate;
- holdout: only when `big_decision` is True and it hasn't been looked at yet.

In [ ]:
def allowed_to_look(split, looks_so_far, big_decision=False):
    """The rule for each split: dev any time; test once per candidate; holdout once, for a big decision."""
    # TODO
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_dev_any_time():
    assert allowed_to_look("dev", 50)


def test_test_once():
    assert allowed_to_look("test", 0) and not allowed_to_look("test", 1)


def test_holdout_needs_a_big_decision():
    assert not allowed_to_look("holdout", 0) and allowed_to_look("holdout", 0, big_decision=True)
    assert not allowed_to_look("holdout", 1, big_decision=True)

Now use it. `look` counts earlier looks **from MLflow itself** (runs tagged with that split and candidate),
refuses if the rule says no, and otherwise opens a run tagged as a look.

The **candidate** here is the set of prepared answers as they are now. The final report on **test** is
their pass rate under the reference judge, with a lesson-level interval: the number you'd publish.

In [ ]:
def look(split, candidate, purpose, big_decision=False, reason=""):
    """Open a run for one look at a split, if the rule allows it; every look is recorded as tags."""
    earlier = mlflow.search_runs(filter_string=f"tags.split = '{split}' and tags.candidate = '{candidate}' and tags.look_kind = 'look'")
    if not allowed_to_look(split, len(earlier), big_decision):
        why = (f"{split} was already looked at {len(earlier)} time(s) for {candidate}" if earlier.shape[0]
               else "holdout is only for a big decision: pass big_decision=True and a reason")
        raise PermissionError(why)
    started = mlflow.start_run(run_name=f"look {len(earlier) + 1} at {split} · {candidate} · {purpose}")
    mlflow.set_tags({"split": split, "candidate": candidate, "look_kind": "look", "look_number": str(len(earlier) + 1),
                     "purpose": purpose, **({"holdout_reason": reason} if split == "holdout" else {}),
                     "mlflow.note.content": (
                         f"**Question:** {purpose}: how do {candidate} score on the {split} split?\n\n"
                         f"**What ran:** look number {len(earlier) + 1} at {split}, allowed by the rule in section 16 "
                         "(dev any time, test once per candidate, holdout once for a big decision"
                         + (f"; reason: {reason}" if reason else "") + ").\n\n"
                         "**How it's measured:** pass rate with a lesson-level bootstrap interval.\n\n"
                         "**Reproduce:** section 16 of notebooks/learning/evaluating_the_tutor.ipynb.")})
    return started


def split_rows(split):
    rows = []
    for lid, items in CHIPS.items():
        if frozen.get(lid) != split:
            continue
        for c in items:
            rec = ANSWERS[lid][c["key"]]
            if rec.get("q") == c["q"] and rec.get("a"):
                rows.append({"lesson_id": lid, "key": c["key"], "passed": bool(rec["judge"]["pass"])})
    return pd.DataFrame(rows)


CANDIDATE = "prepared-answers-v1"
test = split_rows("test")
with look("test", CANDIDATE, purpose="final report"):
    rate = test.passed.mean()
    lo, hi = cluster_bootstrap(test, lambda d: d.passed.mean())
    mlflow.log_params({"split": "test", "answers": len(test), "lessons": test.lesson_id.nunique(), "judged_by": REC["params"]["reference_judge"]})
    mlflow.log_metrics({"pass_rate": rate, "pass_rate_ci_low": lo, "pass_rate_ci_high": hi})
print(f"TEST, reported once: {rate:.1%} of {len(test)} prepared answers pass (lesson bootstrap {lo:.1%} to {hi:.1%})")

In [ ]:
try:
    look("test", CANDIDATE, purpose="just checking again")
except PermissionError as e:
    print("refused:", e)

try:
    look("holdout", CANDIDATE, purpose="curious")
except PermissionError as e:
    print("refused:", e)

untouched = mlflow.search_runs(filter_string="tags.split = 'holdout'")
print(f"runs that ever looked at holdout: {len(untouched)}  <- the audit trail shows it's still a fair final check")

**What to look for.** One test look, logged; a second refused; the holdout refused without a big decision
and a reason; and an audit showing **zero** holdout looks. That's the discipline: the rules are kept by the
record, not by good intentions.

For the judge itself, the same pattern applies: tune the judge's instructions on the dev sample (as many
runs as you like), then judge **test** once with the final version, and keep holdout for the decision to
switch monitoring to the cheaper judge. The project's command does this: `--split test` for the report,
`--split holdout --yes-holdout` only for that decision.

*Run this section a second time and the test look will be refused: that's the rule working. Re-run the
"Track it" setup cell to start a fresh throwaway store.*

## 17. The experiment that really matters: does it help learners?

> **In plain words.** A judge is only a stand-in. The real question is whether answers **help people
> learn**. The app already records what tells us: **Got it / Still stuck** on each answer, and whether the
> learner got the **next question right**. So the real experiment runs on live use.

A professional plan for it, written before any data:

| Part | Plan |
|---|---|
| **Question** | Do prepared answers help as much as live answers from the AI? |
| **Groups** | Each learner is randomly assigned, once, to see prepared answers or live ones for the standard questions (the same person always sees the same kind, so they don't compare) |
| **Primary metric** | Share of first answers that get **Got it** |
| **Secondary** | Next question right first time; Still stuck rate; cost per answer |
| **Guardrail** | Stop early if Still stuck rises by more than 10 points |
| **Analysis** | Compare the two groups' rates with intervals; cluster by learner, since one person asks many questions |
| **Decision rule** | Keep prepared answers if Got it is no more than 3 points worse, since they're instant and free |

How many questions would it take? A common rule of thumb (Lehr's rule) for comparing two rates: about
16 x rate x (1 - rate) / difference squared **per group**.

In [ ]:
for got_it, diff in ((0.7, 0.05), (0.7, 0.03), (0.5, 0.05)):
    per_group = 16 * got_it * (1 - got_it) / diff ** 2
    print(f"Got it around {got_it:.0%}, to detect a {diff:.0%} difference: about {per_group:,.0f} answered questions per group")

**What to look for.** Small differences need a lot of traffic. For a learning app with a few users that
means running for a long time, or only looking for bigger differences. That trade-off is the honest
answer, and it's why the cheaper checks above (judges, golden sets) are still worth having.

## 18. Judges from another family: GPT models on Microsoft Foundry

> **In plain words.** Sonnet judged the prepared answers, and Opus drafted them. Both are Claude, and AI
> judges tend to favour answers from their own family (**self-preference bias**). A judge from another
> family, OpenAI's GPT models on Microsoft Foundry, takes that away. They may bring biases of their own,
> so each is tested exactly like Haiku: the same 301 answers, the same judge prompt, the same rule from
> section 0.

Four candidates, chosen to answer four questions:

| Candidate | How it ran | The question it answers |
|---|---|---|
| GPT-5.4 | batch, half price | Does a strong judge from another family agree? |
| GPT-5.4 mini | batch, half price | Is a third of the price good enough? |
| GPT-6.1 sol | inline (no batch for it) | Does the newest flagship judge better? |
| GPT-5.6 luna | inline (no batch for it) | How cheap can a judge be and still pass? |

The batch ones went as several jobs one after another, because a batch deployment's queue holds a
limited number of tokens at once (the cost, speed and quality notebook, section 10, shows the method).
Each was a real **MLflow eval run** in the `daily-drill` experiment, made by the repo's own command:

```bash
uv run drill tutor prepared-eval --split dev --sample 300 --provider foundry --mode batch \
    --judge-model gpt-5.4 --deployment <your gpt-5.4 batch deployment> --record dev_sample_gpt54.json
```

The run wrote the recording replayed here. Each replay below carries the original run's id as a tag, so
you can open the real run, with its traces, prompt version, dataset digest and exact command.

One honest catch first. The rule compares the candidate with **Sonnet**, the judge we suspect of
favouring its own family. So high agreement with Sonnet partly means "agrees with a possibly biased
judge". Only your own grades (section 14) settle who is right. Read the numbers with that in mind.

In [ ]:
CANDIDATES = {"GPT-5.4 (Foundry, batch)": "dev_sample_gpt54.json", "GPT-5.4 mini (Foundry, batch)": "dev_sample_gpt54mini.json",
              "GPT-6.1 sol (Foundry)": "dev_sample_gpt61sol.json", "GPT-5.6 luna (Foundry)": "dev_sample_gpt56luna.json"}
RUNS = {name: json.loads((RECORDINGS / f).read_text()) for name, f in CANDIDATES.items() if (RECORDINGS / f).exists()}
COLS = {"Haiku 5.5 (Databricks)": "haiku"}
for name, r in RUNS.items():
    col = r["params"]["judge_model"]
    verdicts = {(a["lesson_id"], a["key"]): a["passes"] for a in r["answers"]}
    df[col] = [verdicts.get((x.lesson_id, x.key)) for x in df.itertuples()]
    COLS[name] = col
both = df.dropna(subset=list(COLS.values())).copy()
for col in COLS.values():
    both[col] = both[col].astype(bool)
COST = {"Haiku 5.5 (Databricks)": haiku_per_answer, **{n: r["metrics"]["judge_cost_per_answer_usd"] for n, r in RUNS.items()}}
for name, r in RUNS.items():
    m, p = r["metrics"], r["params"]
    print(f"{name:32} {len(r['answers'])} verdicts, ${m['judge_cost_usd']:.2f} in all, served by {p.get('served_model', p['judge_model'])}"
          f" | MLflow run {r['run_id'][:8]}")
print(f"{len(both)} answers judged by all {len(COLS)} candidates")

### 18a. Side by side, against Sonnet

The same measures as before, for both candidates. Recall and specificity carry lesson-level intervals
(section 6), and agreement is weighted back to dev's real mix (section 8).

In [ ]:
def judge_row(name, col, cost):
    d = both.assign(agree=both[col] == both.sonnet)
    c = confusion(list(d[col]), list(d.sonnet))
    r_lo, r_hi = cluster_bootstrap(d, lambda x: confusion(list(x[col]), list(x.sonnet))["recall"])
    s_lo, s_hi = cluster_bootstrap(d, lambda x: confusion(list(x[col]), list(x.sonnet))["specificity"])
    return {"judge": name, "recall": c["recall"], "recall_ci_low": r_lo, "recall_ci_high": r_hi,
            "precision": c["precision"], "specificity": c["specificity"], "specificity_ci_low": s_lo,
            "specificity_ci_high": s_hi, "kappa": kappa(list(d[col]), list(d.sonnet)), "ac1": ac1(list(d[col]), list(d.sonnet)),
            "weighted_agreement": weighted_agreement(d), "cents_per_answer": 100 * cost}


side = pd.DataFrame([judge_row(name, col, COST[name]) for name, col in COLS.items()]).set_index("judge")
side.round(3)

### 18b. Are the differences real? McNemar on every pair, Holm-adjusted

Every candidate judged the **same** answers, so compare them answer by answer (section 12). For each
pair, count where only one of them agreed with Sonnet. Five candidates make ten pairs, and ten tests
find "differences" by luck, so the p-values are **Holm-adjusted** (section 9).

In [ ]:
from itertools import combinations

pairs = []
for (n1, c1), (n2, c2) in combinations(COLS.items(), 2):
    only1 = int(((both[c1] == both.sonnet) & (both[c2] != both.sonnet)).sum())
    only2 = int(((both[c2] == both.sonnet) & (both[c1] != both.sonnet)).sum())
    pairs.append({"pair": f"{n1} vs {n2}", "only_first_agrees": only1, "only_second_agrees": only2, "p": mcnemar(only1, only2)})
pairs = pd.DataFrame(pairs)
pairs["p_holm"] = holm(list(pairs.p))
pairs.sort_values("p_holm").round(4)

### 18c. Against your own grades, if you've given any

This is the comparison that settles it, because no judge is the truth.

In [ ]:
golden = pd.read_csv(golden_path).fillna("") if golden_path.exists() else pd.DataFrame()
if len(golden):
    golden = golden[golden.your_verdict.str.strip().str.lower().isin(["pass", "fail"])]
if len(golden):
    g = golden.merge(both[["lesson_id", "key", "sonnet", *COLS.values()]], on=["lesson_id", "key"])
    you = list(g.your_verdict.str.strip().str.lower() == "pass")
    for judge in ("sonnet", *COLS.values()):
        c = confusion(list(g[judge]), you)
        print(f"{judge:6} vs you on {len(g)}: recall {c['recall']}, specificity {c['specificity']}, AC1 {ac1(list(g[judge]), you):.2f}")
else:
    print("No grades yet: fill in golden_to_label.csv (section 14). Until then, Sonnet stands in for the truth, said out loud.")

### 18d. The rule decides, and MLflow keeps everything

Each GPT candidate gets its own **eval run**, made exactly like Haiku's in section 15: the same dataset, one trace
per answer, its verdicts as feedback, and its metrics with interval ends. Then one more run, the
**comparison run**, keeps the analysis itself:

- the side-by-side table and McNemar's result;
- a chart of agreement against cost;
- the decision;
- the ids of the runs it compared, as tags.

So anyone can audit the choice from MLflow alone.

In [ ]:
def rule_passes(row):
    return row.recall >= 0.8 and row.specificity >= 0.8 and row.recall_ci_low >= 0.7 and row.specificity_ci_low >= 0.7


def replayed(model, verdicts):
    @scorer(name="judge")
    def judge(inputs):
        a = verdicts[(inputs["lesson_id"], inputs["key"])]
        return Feedback(name="passes", value=bool(a["passes"]), rationale=a["notes"] or None,
                        source=AssessmentSource(source_type="LLM_JUDGE", source_id=model))
    return judge


eval_runs = {"Haiku 5.5 (Databricks)": run.info.run_id}
for name, r in RUNS.items():
    model, row = r["params"]["judge_model"], side.loc[name]
    verdicts = {(a["lesson_id"], a["key"]): a for a in r["answers"]}
    mine = [x for x in records if (x["inputs"]["lesson_id"], x["inputs"]["key"]) in verdicts]
    with mlflow.start_run(run_name=f"judge {model} · foundry {r['params']['mode']} · replay · dev-sample-{len(mine)}") as candidate_run:
        mlflow.set_tags({"split": "dev", "purpose": "exploration", "look_kind": "dev", "provider": "Microsoft Foundry",
                         "mode": r["params"]["mode"], "source_run_id": r["run_id"],
                         "decision": "trust for monitoring" if rule_passes(row) else "do not trust yet",
                         "mlflow.note.content": (
                             f"**Question:** can {model} stand in for Sonnet as the judge of the prepared answers?\n\n"
                             f"**What ran:** a replay of MLflow run {r['run_id']} ({r['params'].get('served_model', model)} on Microsoft "
                             f"Foundry, {r['params']['mode']}), on the stratified dev sample; Sonnet's verdicts are the reference.\n\n"
                             f"**How it's measured:** recall and specificity with lesson-level bootstrap intervals, kappa, AC1, "
                             f"agreement re-weighted to dev's mix. Rule: {PLAN['decision_rule']}\n\n"
                             f"**Reproduce:** `{r.get('reproduce', 'see the source run')}`")})
        mlflow.log_params({"judge_model": model, "judge_prompt": r["params"]["judge_prompt"], "reference_judge": r["params"]["reference_judge"],
                           "split": "dev", "dataset": ds.name, "dataset_digest": ds.digest, "price_per_mtok": r["params"]["price_per_mtok"],
                           "served_model": r["params"].get("served_model", model), "decision_rule": PLAN["decision_rule"],
                           "provider": "foundry", "mode": r["params"]["mode"], "deployment": r["params"].get("deployment", model)})
        mlflow.genai.evaluate(data=mine, predict_fn=answer_for, scorers=[replayed(model, verdicts), word_count])
        mlflow.log_metrics({k: float(v) for k, v in row.items() if v is not None})
        mlflow.log_metrics({k: float(v) for k, v in r["metrics"].items() if k.startswith("judge_")})
        mlflow.log_dict(PLAN, "plan.json")
        c = confusion(list(both[COLS[name]]), list(both.sonnet))
        mlflow.log_table({"judge": ["FAIL", "PASS"], "reference FAIL": [c["caught"], c["missed"]],
                          "reference PASS": [c["false_alarm"], c["let_through"]]}, "confusion_matrix.json")
    eval_runs[name] = candidate_run.info.run_id

fig, ax = plt.subplots(figsize=(6, 4))
for name, r in side.iterrows():
    ax.plot(r.cents_per_answer, r.weighted_agreement, "o" if rule_passes(r) else "x")
    ax.annotate(name, (r.cents_per_answer, r.weighted_agreement), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.scatter([100 * sonnet_per_answer], [1.0], marker="x")
ax.annotate("Sonnet (the reference)", (100 * sonnet_per_answer, 1.0), textcoords="offset points", xytext=(-40, -14), fontsize=8)
ax.set_xlabel("cents per judged answer")
ax.set_ylabel("agreement with Sonnet, weighted to dev's mix")
ax.set_xscale("log")
ax.set_title("Judges on the same answers (o passes the rule, x doesn't)")

with mlflow.start_run(run_name=f"compare {len(COLS)} judges · same {len(both)} prepared answers · dev-sample") as compare_run:
    mlflow.set_tags({"split": "dev", "purpose": "comparison", "look_kind": "dev",
                     "mlflow.note.content": (
                         f"**Question:** which judge should check the tutor's prepared answers every day?\n\n"
                         f"**What ran:** {len(COLS)} candidate judges ({', '.join(COLS)}) on the same {len(both)} prepared answers, "
                         "each its own eval run (tag compared_runs); Sonnet's verdicts are the reference.\n\n"
                         "**How it's measured:** the side-by-side table (recall, specificity with intervals, kappa, AC1, weighted "
                         "agreement, cost), McNemar on every pair with Holm's adjustment, and agreement against cost. "
                         f"Rule: {PLAN['decision_rule']}\n\n"
                         "**Reproduce:** run notebooks/learning/evaluating_the_tutor.ipynb, section 18; each candidate's own run "
                         "has its command."),
                     "compared_runs": ",".join(eval_runs.values()),
                     "decision": ", ".join(f"{n}: {'trust' if rule_passes(r) else 'not yet'}" for n, r in side.iterrows())})
    mlflow.log_params({"dataset": ds.name, "dataset_digest": ds.digest, "reference_judge": REC["params"]["reference_judge"],
                       "test": "McNemar on agreement with the reference, same answers"})
    mlflow.log_params({"candidates": len(COLS), "answers": len(both), "multiple_tests": "Holm"})
    mlflow.log_table(side.reset_index().round(4).to_dict(orient="list"), "side_by_side.json")
    mlflow.log_table(pairs.round(4).to_dict(orient="list"), "mcnemar_pairs_holm.json")
    mlflow.log_dict(eval_runs, "eval_runs.json")
    mlflow.log_figure(fig, "agreement_vs_cost.png")
    mlflow.log_dict(PLAN, "plan.json")
plt.show()

judges = mlflow.search_runs(filter_string="tags.purpose = 'exploration' and tags.split = 'dev'", order_by=["attributes.start_time"])
judges[["tags.mlflow.runName", "params.judge_model", "metrics.recall", "metrics.specificity", "metrics.kappa", "metrics.ac1", "tags.decision"]].round(3)

**What to look for.**

- **The decision tags.** Which candidates pass the rule from section 0?
- **The Holm-adjusted p-values.** Which gaps between candidates are real, and which are luck on 301
  answers?
- **Cost against agreement** (the chart's cost axis is logarithmic). Of the candidates that pass, the
  cheapest is the one to run every day.

Whichever passes, the next step is section 16's discipline. Judge **test** once with it, then use it to
judge new prepared answers. Then the drafter (Opus, Claude) and the judge (GPT) are from different
families.

### 18e. The recommendation

> **In plain words.** Every number above is in MLflow. This cell reads the candidates' runs back from
> there, not from this notebook's variables, so the recommendation rests on what was logged. It applies
> the rule written in section 0, in order, says why each candidate passed or didn't, and logs the
> recommendation to the comparison run.

In [ ]:
from IPython.display import Markdown, display

logged = mlflow.search_runs(filter_string=f"attributes.run_id IN ({', '.join(repr(r) for r in eval_runs.values())})")
logged = logged.set_index(logged["tags.mlflow.runName"])
cols = {"metrics.recall": "recall", "metrics.recall_ci_low": "recall_low", "metrics.specificity": "specificity",
        "metrics.specificity_ci_low": "specificity_low", "metrics.kappa": "kappa", "metrics.ac1": "ac1",
        "metrics.weighted_agreement": "agreement", "metrics.cents_per_answer": "cents"}
board = logged[list(cols)].rename(columns=cols)
board["name"] = [next(n for n, rid in eval_runs.items() if rid == r) for r in logged["run_id"]]
board = board.set_index("name")
board["passes_rule"] = [row.recall >= .8 and row.specificity >= .8 and row.recall_low >= .7 and row.specificity_low >= .7
                        for row in board.itertuples()]
board["closest"] = board[["recall", "specificity"]].min(axis=1)      # the weaker of the two: how far from the bar


def same_as(a, b):
    """McNemar after Holm: can't tell the two apart on these answers?"""
    hit = pairs[pairs.pair.isin([f"{a} vs {b}", f"{b} vs {a}"])]
    return bool(len(hit)) and hit.p_holm.iloc[0] >= 0.05


passing = board[board.passes_rule].sort_values("cents")
lines = [f"**Rule (written first):** {PLAN['decision_rule']}", ""]
if len(passing):
    best = passing.index[0]
    lines.append(f"**Recommendation: {best}** as the daily judge. It passes the rule, and it's the cheapest that does "
                 f"({passing.cents.iloc[0]:.3f} cents an answer).")
    for other in passing.index[1:]:
        lines.append(f"- {other} passes too, at {passing.loc[other, 'cents']:.3f} cents"
                     + (": McNemar can't tell it apart from the winner, so cost decides." if same_as(best, other) else "."))
    lines.append("")
    lines.append("**Before switching:** judge **test** once with it (section 16), and check it against your own grades (section 14).")
else:
    near = board.sort_values("closest", ascending=False).index[0]
    lines.append("**Recommendation: don't switch yet.** No candidate meets the rule against Sonnet.")
    lines.append(f"- Closest: **{near}** (recall {board.loc[near, 'recall']:.2f}, specificity {board.loc[near, 'specificity']:.2f}).")
    lines.append("- The candidates disagree with Sonnet in opposite directions: the strict ones fail most good answers, the "
                 "lenient ones miss real failures. When every judge disagrees with the reference, the reference itself is in "
                 "question: Sonnet is Claude judging Claude's drafts.")
    lines.append("- **Next step:** grade the golden set (section 14, about 60 answers), then score every candidate *and Sonnet* "
                 "against your grades. Whichever judge is closest to you, within the rule, is the one to use. Until then, keep "
                 "the current judge.")
lines.append("")
lines.append("| Candidate | Recall | Specificity | Kappa | AC1 | Cents an answer | Passes the rule |")
lines.append("|---|---|---|---|---|---|---|")
for name, r in board.sort_values("cents").iterrows():
    lines.append(f"| {name} | {r.recall:.2f} (from {r.recall_low:.2f}) | {r.specificity:.2f} (from {r.specificity_low:.2f}) | "
                 f"{r.kappa:.2f} | {r.ac1:.2f} | {r.cents:.3f} | {'yes' if r.passes_rule else 'no'} |")
recommendation = "\n".join(lines)
display(Markdown(recommendation))

with mlflow.start_run(run_id=compare_run.info.run_id):
    mlflow.log_text(recommendation, "recommendation.md")
    mlflow.set_tag("recommendation", passing.index[0] if len(passing) else "don't switch yet: grade the golden set")
    mlflow.log_table(board.reset_index().round(4).to_dict(orient="list"), "scoreboard.json")

# The aliases. Every judge setup is a version of the registered prompt tutor_judge, with its model attached;
# champion is the judge in use, and challenger is set only when a candidate passed the rule on dev.
champion = tpe.ensure_champion(mlflow)
for rid in eval_runs.values():
    tpe.link_judge(mlflow, rid)                          # each run records the judge version it tested
if len(passing):
    named = tpe.set_judge_alias(mlflow, "challenger", eval_runs[passing.index[0]], "passed the rule on dev; the cheapest that does")
    print(f"challenger -> {named}. Promote it to champion only after its look at test and your grades.")
else:
    try:
        mlflow.genai.delete_prompt_alias(champion.name, "challenger")
    except Exception:
        pass
    print(f"No challenger: nothing passed the rule. champion stays {champion.name}/{champion.version} "
          f"({champion.model_config['model_name']}).")
for name, rid in eval_runs.items():
    v = mlflow.get_run(rid).data.params.get("judge_version")
    print(f"  {name:32} tested {v}")
real = {n: r["run_id"] for n, r in RUNS.items()}
if len(passing) and passing.index[0] in real:
    print(f"\nIn your own MLflow (where the real runs live): uv run drill tutor judge-alias challenger --run {real[passing.index[0]]} "
          f"--reason \"passed the rule on dev\"")

**What to look for.** The recommendation follows from the rule, not from the chart. If you disagree with
it, change the rule in section 0, say why, and run again. Never change the result. Every run it read is
linked from the comparison run (tag `compared_runs`).

**The aliases.** A judge is a prompt plus a model, so each judge setup is a version of the registered
prompt `tutor_judge`, with its model attached. Each judge run records the version it tested
(`judge_version`).

- `champion` is the judge in use.
- `challenger` is set only when a candidate passes the rule on dev.
- A challenger becomes champion only after its one look at **test** (section 16) and a check against your
  grades: `uv run drill tutor judge-alias champion --run <its test run> --yes-promote --reason "..."`.

The course's lesson "Model registry and promotion" teaches the same pattern for models.

### 18f. Where they disagree, check by check

> **In plain words.** "Pass" means all five checks passed, and the rule in section 0 is set on that one
> verdict, because pass or fail is the only thing the app acts on. But every verdict carries the five
> checks underneath. Looking at them one at a time shows *where* the judges split, and whether the
> split is the judges' fault or the checklist's.

In [ ]:
per_check = []
for name, r in RUNS.items():
    items = [a for a in r["answers"] if tpe.CHECKS[0] in a]
    for c in tpe.CHECKS:
        sonnet_fail = sum(not ANSWERS[a["lesson_id"]][a["key"]]["judge"][c] for a in items) / len(items)
        judge_fail = sum(not a[c] for a in items) / len(items)
        agree = sum(a[c] == ANSWERS[a["lesson_id"]][a["key"]]["judge"][c] for a in items) / len(items)
        per_check.append({"judge": name, "check": c, "sonnet_fails": sonnet_fail, "judge_fails": judge_fail, "agree": agree})
h = REC["metrics"]   # the Haiku run kept per-check rates, not per-answer checks
for c in tpe.CHECKS:
    per_check.append({"judge": "Haiku 5.5 (Databricks)", "check": c, "sonnet_fails": float("nan"), "judge_fails": 1 - h[c + "_rate"], "agree": h[c + "_agreement"]})
per_check = pd.DataFrame(per_check)
fails = per_check.pivot(index="check", columns="judge", values="judge_fails").reindex(tpe.CHECKS)
fails.insert(0, "Sonnet (reference)", per_check[per_check.judge != "Haiku 5.5 (Databricks)"].groupby("check").sonnet_fails.first().reindex(tpe.CHECKS))
agrees = per_check.pivot(index="check", columns="judge", values="agree").reindex(tpe.CHECKS)
print("share of answers each judge FAILS on each check:")
display((100 * fails).round(0).astype(int).astype(str) + "%")
print("agreement with Sonnet on each check:")
display((100 * agrees).round(0).astype(int).astype(str) + "%")

with mlflow.start_run(run_id=compare_run.info.run_id):
    mlflow.log_table(per_check.round(4).to_dict(orient="list"), "per_check.json")
    worst = agrees.min(axis=1).sort_values().index[:2].tolist()
    mlflow.set_tag("checks_in_dispute", ",".join(worst))
print("checks in dispute (lowest agreement):", worst)

**What to look for.** Three checks are not in dispute: on *answers it*, *no giveaway* and *standalone*
every judge agrees with Sonnet almost all the time. The split sits in two checks, and each tells a
different story.

- **Plain** is a taste judgement with no definition. Some judges fail nearly half the answers on it,
  others almost none. That is the checklist's fault. The server already has code checks for the word
  limit, maths notation and leaked exercise code (`drill.server.observe.answer_checks`, the `rules`
  scorer in the real runs), so "plain" can become a rule no judge reads two ways: a word limit plus a
  list of terms that must be explained.
- **Correct** is the check that matters most, and every candidate is stricter than Sonnet on it. Read
  their notes: they fail answers that add true material the lesson does not contain ("goes beyond
  using only the lesson as truth"), while Sonnet reads the same instruction as "nothing contradicts the
  lesson". The checklist says both at once. Decide which you mean; either is fine.

**What that means for the rule.** One bar on the overall verdict is blunt. The course's own must-pass
idea (lesson "Must-pass gate") fits better: *correct* and *no giveaway* are the checks a wrong verdict
hurts on, so they carry the 80% bar on their own; *plain* moves to code. Write that into section 0's
plan before the next round, then re-judge with the clearer checklist. The judges that looked strict
or lenient may turn out to agree once the checklist says one thing.

### 18g. Checklist v2: does one meaning per check make the judges agree?

> **In plain words.** Section 18f blamed the checklist, not the judges: "plain" had no definition and
> "correct" said two things at once. Checklist 2 (`tutor_prepared.JUDGE2`) says each thing once:
> *correct* becomes **no contradiction** (must-pass) and **within the lesson** (reported, and you decide
> below whether it blocks); *answers it* is tied to the question's own instructions; **concise** and
> **tone** are added; and *plain* leaves the judge for code (`drill.answer_quality`: a readability grade,
> fillers, repeated sentences, terms used unexplained, numbers and names outside the lesson).
>
> The test of a clearer checklist is not a higher pass rate. It is **agreement**: two judges reading the
> same words should reach the same verdicts more often than before. The two cheapest judges re-graded the
> same 301 answers with checklist 2 (`drill tutor prepared-eval --judge-version 2`, two eval runs).

In [ ]:
V1 = {"GPT-5.4 mini": "dev_sample_gpt54mini.json", "GPT-5.6 luna": "dev_sample_gpt56luna.json"}
V2 = {"GPT-5.4 mini": "dev_sample_gpt54mini_v2.json", "GPT-5.6 luna": "dev_sample_gpt56luna_v2.json"}
load = lambda f: {(a["lesson_id"], a["key"]): a for a in json.loads((RECORDINGS / f).read_text())["answers"]}
v1, v2 = {n: load(f) for n, f in V1.items()}, {n: load(f) for n, f in V2.items()}
shared = sorted(set.intersection(*[set(d) for d in (*v1.values(), *v2.values())]))
sonnet = {k: v2["GPT-5.4 mini"][k]["reference_pass"] for k in shared}   # the reference verdict travels with every recording
agreement = []
for version, recs, checks in (("v1", v1, tpe.CHECKS), ("v2", v2, tp.CHECKS2)):
    a, b = (recs[n] for n in V1)
    for c in ("pass", *checks):
        key = "passes" if c == "pass" else c
        agreement.append({"checklist": version, "check": c, "judges_agree": sum(a[k][key] == b[k][key] for k in shared) / len(shared),
                          **{f"{n} fails": sum(not recs[n][k][key] for k in shared) / len(shared) for n in V1}})
agreement = pd.DataFrame(agreement)
print(f"{len(shared)} answers graded by both judges under both checklists")
display(agreement.pivot(index="check", columns="checklist", values="judges_agree").round(2).rename(columns=lambda c: f"judges agree ({c})"))
vs_sonnet = []
for version, recs in (("v1", v1), ("v2", v2)):
    for n in V1:
        c = confusion([recs[n][k]["passes"] for k in shared], [sonnet[k] for k in shared])
        vs_sonnet.append({"checklist": version, "judge": n, "catches Sonnet's fails": c["recall"], "keeps Sonnet's passes": c["specificity"],
                          "kappa with Sonnet": kappa([recs[n][k]["passes"] for k in shared], [sonnet[k] for k in shared])})
vs_sonnet = pd.DataFrame(vs_sonnet).set_index(["checklist", "judge"])
display(vs_sonnet.round(2))
# what the code checks find on the same answers, computed here with the current rules (the recordings keep an older pass)
from drill import answer_quality as aq
glossary = aq.glossary_terms(DATA)
by_chip = {(lid, c["key"]): c for lid, items in CHIPS.items() for c in items}
q = pd.DataFrame([aq.quality_checks(ANSWERS[k[0]][k[1]]["a"], tp._context(DATA, k[0]) + "\n" + by_chip[k]["q"] + "\n" + by_chip[k]["how"], glossary)
                  for k in shared])
code_found = {"hard_to_read": q.hard_to_read.mean(), "repeated": q.repeated.mean(), "fillers": q.fillers.map(bool).mean(),
              "unexplained_terms": q.unexplained_terms.map(bool).mean(), "unknown_numbers": q.unknown_numbers.map(bool).mean(),
              "unknown_names": q.unknown_names.map(bool).mean()}
print("share of answers the code checks flag:", {k: f"{v:.0%}" for k, v in code_found.items()})
with mlflow.start_run(run_id=compare_run.info.run_id):
    mlflow.log_table(agreement.round(4).to_dict(orient="list"), "checklist_v1_vs_v2_agreement.json")
    mlflow.log_table(vs_sonnet.reset_index().round(4).to_dict(orient="list"), "checklist_v1_vs_v2_vs_sonnet.json")
    mlflow.log_dict(code_found, "code_checks_found.json")
    mlflow.log_metrics({f"v2_judges_agree_{r.check}": r.judges_agree for r in agreement[agreement.checklist == "v2"].itertuples()})

**What to look for.** Compare the `pass` row first: did the two judges agree with each other more under
v2? Then the checks that were in dispute. If *no contradiction* agrees far better than *correct* did, the
split worked. *Within the lesson* is new, so its row says how often the judges think an answer adds
material, which is the question for you below. Against Sonnet, remember Sonnet graded with checklist 1
and may read "correct" its own way: a lower catch rate under v2 can mean the v2 judge is right.

### 18h. Your call: may an answer add true material the lesson doesn't state?

> **In plain words.** Checklist 2 reports *within the lesson* but doesn't block on it, because that is a
> policy, not a fact, and it is yours to set. Set it on examples, not in the abstract. The cell below
> writes a sheet of answers the v2 judges flagged as going beyond the lesson, with the judge's note naming
> the addition. Mark each one in the `your_verdict` column:
>
> - `true and helpful`: the addition is right and a learner is better off for it;
> - `true but off the lesson`: right, but it pulls the learner away from what the lesson teaches;
> - `wrong`: the addition is false, or misstates a number or a name;
> - `not an addition`: the judge was wrong to flag it.
>
> **The rule, written before you look:** block on *within the lesson* if at least a fifth of the real
> additions are `wrong`. Below that, allow them and keep reporting. Either way, your labels become human
> feedback on the answers' traces in MLflow, next to the judges' verdicts.

In [ ]:
RULE_WRONG_SHARE = 0.2
additions_path = Path("additions_to_label.csv")
flagged = [k for k in shared if any(not v2[n][k]["within_lesson"] for n in V2)]
if not additions_path.exists():
    pick = random.Random(7).sample(flagged, min(40, len(flagged)))
    sheet = pd.DataFrame([{"lesson_id": k[0], "key": k[1], "question": ANSWERS[k[0]][k[1]]["q"], "answer": ANSWERS[k[0]][k[1]]["a"],
                           **{f"note ({n})": v2[n][k]["notes"] for n in V2 if not v2[n][k]["within_lesson"]},
                           "your_verdict": ""} for k in pick])
    sheet.to_csv(additions_path, index=False)
    print(f"{len(flagged)} answers flagged as going beyond the lesson; wrote {len(sheet)} of them to {additions_path.resolve()}")
else:
    print(f"{additions_path} already exists: fill in your_verdict, then run the next cell")

In [ ]:
CHOICES = ("true and helpful", "true but off the lesson", "wrong", "not an addition")
labels = pd.read_csv(additions_path).fillna("") if additions_path.exists() else pd.DataFrame()
if len(labels):
    labels["your_verdict"] = labels.your_verdict.str.strip().str.lower()
    labels = labels[labels.your_verdict.isin(CHOICES)]
if len(labels):
    counts = labels.your_verdict.value_counts().reindex(CHOICES, fill_value=0)
    real = labels[labels.your_verdict != "not an addition"]
    wrong_share = (real.your_verdict == "wrong").mean() if len(real) else float("nan")
    block = bool(len(real)) and wrong_share >= RULE_WRONG_SHARE
    print(counts.to_string())
    print(f"\nof {len(real)} real additions, {wrong_share:.0%} are wrong -> WITHIN_LESSON_BLOCKS = {block}"
          f" (the rule: block at {RULE_WRONG_SHARE:.0%} or more)")
    print(f"the judges' flag was right {1 - counts['not an addition'] / len(labels):.0%} of the time")
    with mlflow.start_run(run_name=f"human labels · additions beyond the lesson · dev-sample-{len(labels)}") as labels_run:
        mlflow.set_tags({"split": "dev", "purpose": "human labels", "source": "a person",
                         "decision": f"WITHIN_LESSON_BLOCKS = {block}",
                         "mlflow.note.content": (
                             "**Question:** may a prepared answer add true material the lesson doesn't state?\n\n"
                             f"**What ran:** a person labelled {len(labels)} answers the checklist-2 judges flagged as going beyond the lesson "
                             "(true and helpful / true but off the lesson / wrong / not an addition).\n\n"
                             f"**How it's measured:** the share of real additions that are wrong; the rule, written first, blocks at {RULE_WRONG_SHARE:.0%}.\n\n"
                             "**Reproduce:** section 18h of notebooks/learning/evaluating_the_tutor.ipynb with the filled-in additions_to_label.csv.")})
        mlflow.log_params({"rule_wrong_share": RULE_WRONG_SHARE, "labelled": len(labels), "flagged": len(flagged)})
        mlflow.log_metrics({"wrong_share": float(wrong_share) if real.shape[0] else 0.0, "real_additions": float(len(real)),
                            **{f"count_{c.replace(' ', '_')}": float(counts[c]) for c in CHOICES}})
        mlflow.log_table(labels.to_dict(orient="list"), "labels.json")
    # the labels onto the answers' traces, as human feedback next to the judges' verdicts
    by_key = {(r.lesson_id, r.key): r.your_verdict for r in labels.itertuples()}
    attached = 0
    for tr in mlflow.search_traces(run_id=run.info.run_id, return_type="list"):
        inputs = json.loads(tr.data.request) if isinstance(tr.data.request, str) else tr.data.request
        verdict = by_key.get((inputs.get("lesson_id"), inputs.get("key")))
        if verdict and not any(a.name == "addition_ok" for a in tr.info.assessments):
            mlflow.log_feedback(trace_id=tr.info.trace_id, name="addition_ok", value=verdict in ("true and helpful", "not an addition"),
                                rationale=verdict, source=AssessmentSource(source_type="HUMAN", source_id="owner"))
            attached += 1
    print(f"{attached} labels attached to the answers' traces as human feedback")
else:
    print("No labels yet: fill in your_verdict in additions_to_label.csv (the choices are above), then run this cell again.")

**What to look for.** Your `wrong` share decides the policy, and the `not an addition` share grades the
judges' flag itself: a flag that is wrong half the time is not worth blocking on, whatever the policy.
Once you have set `WITHIN_LESSON_BLOCKS` in `tutor_prepared.py`, re-run the v2 judges once on the test
split (section 16) before switching drafting to checklist 2 (`drill tutor prepare --judge-version 2`).

## 19. When the model isn't sure: can the drafter's own uncertainty predict a fail?

> **In plain words.** When a model writes a fact it is unsure of, two things can show it. The
> word-level probabilities at that spot are spread out (its **confidence**), and asking the same
> question several times gives answers that disagree (**self-consistency**). If either predicts the
> judge's verdict, it is worth using *before* the judge: flag the unsure draft, send it to a person, or
> write it again. If neither does, we have measured that instead of assuming it.
>
> Two cautions first. Models are often confidently wrong, so a high confidence never proves an answer
> right. And confidence needs the model's internal scores: GPT models on Foundry return them
> (`logprobs`); the Claude endpoints don't, so this runs on GPT-5.4 mini.

The run (`uv run drill tutor uncertainty`, 2026-10-18): 50 dev questions, each drafted 3 times by
GPT-5.4 mini as a batch with every token's log probability, the 150 drafts judged by GPT-5.4 with the
prepared answers' checklist. Four code signals per draft, no model needed:

| Signal | What it is | Expected if the idea holds |
|---|---|---|
| `mean_logprob` | the average log probability of the answer's tokens (`exp(-mean)` is the perplexity) | passes score higher |
| `min_prob` | the least likely token's probability | passes score higher |
| `risky_min_prob` | the least likely token among numbers and names | passes score higher |
| `disagreement` | 1 minus the draft's word overlap with the question's other drafts | fails score higher |
| `numbers_differ` | whether the three drafts state different numbers | fails score higher |

In [ ]:
UNC = json.loads((RECORDINGS / "draft_uncertainty_gpt54mini.json").read_text())
drafts = pd.DataFrame(UNC["drafts"])
drafts = drafts[drafts.passes.notna()].copy()
drafts["passes"] = drafts.passes.astype(bool)
um = UNC["metrics"]
print(f"{len(drafts)} drafts of {UNC['params']['questions']} questions judged; pass rate {drafts.passes.mean():.0%}; "
      f"drafting ${um['draft_cost_usd']:.2f}, judging ${um['judge_cost_usd']:.2f} | MLflow run {UNC['run_id'][:8]}")
print(f"questions whose 3 drafts all got the same verdict: {um['questions_with_unanimous_verdicts']:.0%}")
drafts[["lesson_id", "key", "sample", "mean_logprob", "min_prob", "risky_min_prob", "disagreement", "numbers_differ", "passes"]].head(6).round(3)

**Exercise.** How do you score a signal that has no threshold yet? With a **rank-based AUC**: the
chance that a randomly chosen passing draft scores higher than a randomly chosen failing one. 0.5 means
the signal is useless (a coin flip), 1 means perfect, below 0.5 means it points the wrong way. Compare
every passing score with every failing score: count a win for each pair where the pass is higher, half
a win for a tie, then divide by the number of pairs. Return `None` if either group is empty.

In [ ]:
def rank_auc(scores, passed):
    """The chance a passing draft scores higher than a failing one (ties count half)."""
    # TODO: pos = scores of the passes, neg = scores of the fails; wins over every pair; wins / (len(pos) * len(neg))
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_perfect_signal():
    assert rank_auc([0.9, 0.8, 0.2, 0.1], [True, True, False, False]) == 1.0


def test_no_signal():
    assert rank_auc([0.5, 0.5, 0.5, 0.5], [True, True, False, False]) == 0.5


def test_backwards_signal():
    assert rank_auc([0.1, 0.2, 0.8, 0.9], [True, True, False, False]) == 0.0


def test_one_empty_group():
    assert rank_auc([0.1, 0.2], [True, True]) is None

In [ ]:
SIGNALS = {"mean_logprob": 1, "min_prob": 1, "risky_min_prob": 1, "disagreement": -1, "numbers_differ": -1}   # -1: higher means worse
power = pd.DataFrame([{"signal": s, "auc": rank_auc(list(sign * drafts[s]), list(drafts.passes)),
                       "mean_when_pass": drafts[drafts.passes][s].mean(), "mean_when_fail": drafts[~drafts.passes][s].mean()}
                      for s, sign in SIGNALS.items()]).set_index("signal")
# the simplest use: flag the least confident third. What is the pass rate inside the flag, and outside it?
third = drafts.mean_logprob.quantile(1 / 3)
flagged = drafts.mean_logprob <= third
power.loc["mean_logprob", "pass_rate_flagged_third"] = drafts[flagged].passes.mean()
power.loc["mean_logprob", "pass_rate_rest"] = drafts[~flagged].passes.mean()
power.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(list(power.index), list(power.auc))
ax.axvline(0.5, linestyle=":", linewidth=1)
ax.set_xlim(0, 1)
ax.set_xlabel("AUC: chance a passing draft looks more confident than a failing one")
ax.set_title("Does the drafter's uncertainty predict the judge?")
plt.tight_layout()

# the decision rule, written before looking: use a signal only if its AUC is at least 0.7
RULE_AUC = 0.7
useful = power[power.auc >= RULE_AUC]
print("signals worth a drafting-time flag:", list(useful.index) if len(useful) else "none")

with mlflow.start_run(run_name=f"draft uncertainty · {UNC['params']['draft_model']} x{UNC['params']['samples']} · replay · dev-{UNC['params']['questions']}") as unc_run:
    mlflow.set_tags({"split": "dev", "purpose": "exploration", "source_run_id": UNC["run_id"], "provider": "Microsoft Foundry",
                     "decision": ("flag on " + ", ".join(useful.index)) if len(useful) else "no useful signal",
                     "mlflow.note.content": (
                         "**Question:** does the drafter's own uncertainty predict a judge fail?\n\n"
                         f"**What ran:** a replay of MLflow run {UNC['run_id']} ({UNC['params']['questions']} dev questions x "
                         f"{UNC['params']['samples']} drafts by {UNC['params']['draft_model']} with token log probabilities, judged by "
                         f"{UNC['params']['judge_model']}).\n\n"
                         f"**How it's measured:** a rank-based AUC per signal against the verdict; a signal is used only if AUC >= {RULE_AUC}.\n\n"
                         f"**Reproduce:** `{UNC['reproduce']}`")})
    mlflow.log_params({"draft_model": UNC["params"]["draft_model"], "judge_model": UNC["params"]["judge_model"],
                       "questions": UNC["params"]["questions"], "samples": UNC["params"]["samples"], "rule_auc": RULE_AUC})
    mlflow.log_metrics({f"auc_{s}": float(v) for s, v in power.auc.items() if v is not None})
    mlflow.log_metrics({"pass_rate": float(drafts.passes.mean()), "unanimous_questions": um["questions_with_unanimous_verdicts"]})
    mlflow.log_table(power.reset_index().round(4).to_dict(orient="list"), "predictive_power.json")
    mlflow.log_table(drafts.drop(columns=["text"]).round(4).to_dict(orient="list"), "drafts.json")
    mlflow.log_figure(fig, "auc.png")
plt.show()

**What to look for.**

- **The AUCs.** Anything near 0.5 is noise. A signal is only worth wiring in if it clears the rule
  (0.7), *and* the flagged third's pass rate is clearly lower than the rest's. Otherwise the flag sends
  good answers to a person for nothing.
- **Confidence against disagreement.** If disagreement predicts better than any confidence number, the
  cheap habit is "write it three times and compare", which works on any model, Claude included.
- **The numbers.** `numbers_differ` is the signal to watch for a tutor: when three drafts state
  different numbers, at least two are made up.
- **Unanimity.** Questions whose three drafts all got the same verdict tell you the verdict is about
  the question, not the draft. The others are where a second attempt would change the outcome.

**Where a useful signal would go:** not into the judge, which reads the finished answer, but into
`drill tutor prepare`, at drafting time: a draft below the threshold is written again or sent to a
person before it is ever judged. The threshold would be set on this dev run and checked once on test.

## What you used

| Idea | Course lesson | In this notebook |
|---|---|---|
| Write the plan first | Write the plan before the data | section 0 |
| Fair splits, stratify, golden set | Dev, test and holdout sets | sections 2, 3, 14 |
| Splits that never move | Splits that don't move | section 2 |
| How many examples | How many examples do you need? | sections 5, 12, 15 |
| Pass rate with its margin | How sure is that pass rate? | section 5 |
| Agreement beyond chance | Do two judges agree? | section 7 |
| Real win or luck | Real win or luck | section 12 |
| Must-pass gate | Must-pass gate | section 0's decision rule, section 15's decision tag |
| Dev, test and holdout, used properly | Dev, test and holdout sets | section 16 |
| Quality, cost and speed together | Choosing a model | section 13 |
| Does the judge agree with itself? | Does the judge agree with itself? | sections 11, 19 |

Beyond the course: the confusion matrix, the lesson-level bootstrap and design effect, Gwet's AC1, sample
weights, the Holm adjustment, length bias, test-retest consistency, power by simulation, and an online
experiment plan.